# Project 2 Craiglist NYC listings scraped Dataset, Cleaning, integration and Analysis

# Imports and Loading the data

## 1. Imports

We import the following libraries:

In [ ]:
import numpy  as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as  plt
import re
from scipy import stats
from scipy.stats import pearsonr
from rapidfuzz import process, fuzz

## 2. Loading the Data

We load the raw CSV with low_memory=False to avoid dtype inference warnings on mixed-type columns. and then we use head() to get the first five values and finally we drop duplicates.

In [ ]:
df1 = pd.read_csv('nyc_apartments.csv')
df=pd.read_csv('airbnb_cleaned.csv')
# First thing we do is we check the nulls in the dataset
df1.isnull().sum() # The scraped dataset has no nulls

In [ ]:
df1.head() # check the first 5 rows of the dataset to see what we are workingg with

There are mainly a couple problems that we can see within the dataset which are as follows:
1. URL is a useless column , can't really analyze anything with it.
2. Area is encoded in shortcuts such as Manhattan being written as mnh
3. Price is formatted as a \\$Value String
4. Location which can be used to extract neighbourhood group is displayed as Place/Place
5. There's no indication of roomtype in craiglist that can be easily infered from these columns

# Preprocessing

## Cleaning Columns and Rows

In [ ]:
# we'll drop the url column as its uselesss for our analysis
df1=df1.drop(columns=['url'])
df1=df1.drop_duplicates()
df1 = df1.drop_duplicates(subset='title')  # Assuming that instances with the same 'title' value are duplicates of the same listing, so they will be dropped
df1.info()

> As craiglist is a "everyone lists anything website", many of its neighbourhood values aren't necessarily as accurate as it should be we'll be using neighbourhoods from our AirBNB - naturally a more accurate website as it's main functionality is to rent properties - dataset as a reference for all the possible neighbourhood values

### Dealing with area column

In [ ]:
# we'll be using AirBNB dataset as a reference for all possible neighbourhoods
all_locations=df.neighbourhood.unique()
all_locations

In [ ]:
df1['area'].unique() # seems like we have mnh, brk, que , brx, stn as neighbourhood group values which is correct
# we'll just map to a borough

In [ ]:
# Mapping each area code to a full Borough name
borough_map = {
    'mnh': 'Manhattan',
    'brk': 'Brooklyn',
    'que': 'Queens',
    'brx': 'Bronx',
    'stn': 'Staten Island'
}
df1['borough'] = df1['area'].map(borough_map)
#drop old area column as it's no longer needed
df1 = df1.drop(columns=['area'])

In [ ]:
df1.head() # lets take another look at the dataset

### Dealing with Price column
Now it's time to deal with Price, it's written in \\$ Value format, so a simple regex that just removes '\\$' and ',' and convert to float 

In [ ]:
df1['price'] = df1['price'].astype(str).str.replace(r'[\$,]', '', regex=True).str.strip().astype(float) # clean price

### Dealing with Room types

Classifies listing title into room type: Hotel / Shared / Entire home/apt / Private / Unknown

In [ ]:
def extract_room_type(title):
    # Classifies listing title into room type: Hotel / Shared / Entire home/apt / Private / Unknown
    if pd.isna(title) or not str(title).strip():
        return 'Unknown'
    
    text = str(title).lower()
    
    # Normalize punctuation to space
    text = re.sub(r'[^\w\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    
    # naturally listings have many abbreviations, so these are some common abbreviations. and we replace these abbreviations with the normal w ord 
    abbrev  = {
        r'\bbedrm\b': 'bedroom',
        r'\bbr\b': 'bedroom',
        r'\bbd\b': 'bedroom',
        r'\bbdr\b': 'bedroom',
        r'\bapt\b': 'apartment',
        r'\bstudio\b': 'studio apartment',
    }
    for pattern, repl in abbrev.items():
        text = re.sub(pattern, repl, text)
    
    # Check order
    # we first check if it's a hotel/hostel soo
    if re.search(r'\b(hotel|hostel)\b', text):
        return 'Hotel room'
    # we check if there are "room mates" or if its a dorm or if there's a shared amenity or if the title says anything like "room for rent"
    if re.search(r'\b(shared room|shared bath|roommate|dorm|rooms? 4 rent|rooms? for rent)\b', text):
        return 'Shared room'
    
    # entire home /apt should have at least TWO apartments. Could also be a studio. we check for basically all kinds of apartments and homes
    if re.search(r'\b([2-9]|ten|eleven|twelve)\s*(?:bed|br|bedroom|ba|bath)\b', text) or \
       re.search(r'\b(apartment|studio apartment|loft|duplex|townhouse|house|penthouse|basement)\b', text):
        return 'Entire home/apt'
    
    # private room : a private room will have one bed  obviously 
    if re.search(r'\b1\s*(?:bed|br|bedroom)\b', text) or \
       re.search(r'\b(one)\s*(?:bed|br|bedroom)\b', text) or \
       re.search(r'\b(bedroom|private)\b', text):
        return 'Private room'
    
    return 'Unknown'

In [ ]:
df1['room_type'] = df1['title'].apply(extract_room_type)
#display the results
print('Room Type Distribution:')
print(df1['room_type'].value_counts()) # 422 unknown is still a lot of unknown values which is to be expected as we are extracting this form the title

In [ ]:
unknowns = df1[df1['room_type'] == 'Unknown']['title'].sample(20)
unknowns 

Looking at the unknowns values, we'll asume they are entire home/apartments


In [ ]:
# Visualize room_type distribution
plt.figure(figsize=(8,5))
df1['room_type'].value_counts().plot(kind='bar', color=['skyblue', 'salmon', 'lightgreen', 'orange', 'gray'])
plt.title('Room Type Distribution (Before Converting Unknown)')
plt.xlabel('Room Type')
plt.ylabel('Count')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

# Show the raw counts
print("Counts before conversion:")
print(df1['room_type'].value_counts())

In [ ]:
df1.loc[df1['room_type'] == 'Unknown', 'room_type'] = 'Entire home/apt' # assume unknowns are all entire homes or apts
print(df1['room_type'].value_counts()) 

### Extracting neighbourhood from location
extract_neighbourhood is used to try to extract neighborhood from location

In [ ]:
def extract_neighbourhood(location, valid_locations=all_locations):
    # Extracts neighborhood from location string using fuzzy matching (60% threshold), falls back to 'Others'

    if pd.isna(location):
        return 'Others'
    
    location = str(location).strip().title()
    
    # split
    parts = location.split('/')
    
    left = parts[0].strip().title()
    right = parts[1].strip().title() if len(parts) > 1 else None
    
    def fuzzy_match(val):
        match, score, _ = process.extractOne(
            val,
            valid_locations,
            scorer=fuzz.ratio
        )
        if score >= 60:
            return match
        return None
    
    # try left
    match = fuzzy_match(left)
    if match:
        return match
    
    # try right
    if right:
        match = fuzzy_match(right)
        if match:
            return match
    
    return 'Others'
    
df1['neighbourhood'] = df1['location'].apply(extract_neighbourhood)
df1

In [ ]:
print((df1['neighbourhood'] == 'Others').sum())
df1[df1['neighbourhood'] == 'Others']

198 rows of others will be convered to their borough value

In [ ]:
# Visualize neighbourhood distribution before conversion
# Top 15 most common neighbourhoods 
neigh_counts = df1['neighbourhood'].value_counts()

plt.figure(figsize=(12,6))
neigh_counts.head(15).plot(kind='barh', color='teal')
plt.title('Top 15 Neighbourhoods (Before Converting "Others")')
plt.xlabel('Count')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

In [ ]:
df1.loc[df1['neighbourhood'] == 'Others', 'neighbourhood'] = df1['borough'] # change others to borough

### Drop location column

In [ ]:
df1.drop(columns=['location'])

### Visualize cleaned data

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))
# Room type after converting 'Unknown
room_counts = df1['room_type'].value_counts()
room_counts.plot(kind='bar', ax=ax1, color=['skyblue', 'salmon', 'lightgreen', 'orange', 'gray'])
ax1.set_title('Room Type Distribution (After Converting "Unknown")')
ax1.set_xlabel('Room Type')
ax1.set_ylabel('Count')
ax1.tick_params(axis='x', rotation=45)

# Top 15 neighbourhoods after converting Others to borough
neigh_counts = df1['neighbourhood'].value_counts().head(15)
neigh_counts.plot(kind='barh', ax=ax2, color='coral')
ax2.set_title('Top 15 Neighbourhoods (After Converting "Others" to Borough)')
ax2.set_xlabel('Count')
ax2.invert_yaxis()

plt.tight_layout()
plt.show()


### Save the cleaned data

In [ ]:
df1.to_csv("Craigslist_Cleaned.csv",index=False)

# Integration of Data

In [ ]:
# 1. what we strated with
print("\nSOURCE DATA SIZES:")

print(f"Raw Craigslist data (df1): {len(df1)} listings")
print(f"Airbnb cleaned data (df): {len(df)} listings")

## AirBNB Summaries

### Normalize columns

In [ ]:
df['neighbourhood'] = df['neighbourhood'].astype(str).str.strip().str.title() #normalizing names to account for misspells and extra spaces
df['neighbourhood group'] = df['neighbourhood group'].astype(str).str.strip().str.title()
df1['neighbourhood'] = df1['neighbourhood'].astype(str).str.strip().str.title()
df1['borough'] = df1['borough'].astype(str).str.strip().str.title()

### Neighbourhood stats

In [ ]:
nb_stats = df.groupby('neighbourhood').agg({
    'price': 'mean',
    'id': 'count'
}).rename(columns={'price': 'airbnb_price_nb', 'id': 'airbnb_density_nb'}).reset_index()



### Borough stats

In [ ]:
boro_stats = df.groupby('neighbourhood group').agg({
    'price': 'mean',
    'id': 'count',
    'reviews per month': 'mean',       
    'availability 365': 'mean',        
    'minimum nights': 'mean',          
    'review rate number': 'mean'       
}).rename(columns={
    'price': 'airbnb_price_boro', 
    'id': 'airbnb_density_boro',
    'reviews per month': 'boro_avg_rpm',
    'availability 365': 'boro_avg_avail',
    'minimum nights': 'boro_avg_min_n',
    'review rate number': 'boro_avg_rate'
}).reset_index()


### Room Type stats

In [ ]:
rt_stats = df.groupby('room type').agg({
    'number of reviews': 'mean'
}).rename(columns={'number of reviews': 'avg_reviews'}).reset_index()

### Extra Stats

In [ ]:
nb_extra_stats = df.groupby('neighbourhood').agg({
    'reviews per month': 'mean',
    'availability 365': 'mean',
    'minimum nights': 'mean',
    'review rate number': 'mean'}).rename(columns={
    'reviews per month': 'avg_reviews_per_month',
    'availability 365': 'avg_availability_365',
    'minimum nights': 'avg_min_nights',
    'review rate number': 'avg_review_rate'}).reset_index()

### Merging Stats

In [ ]:
df_final1 = pd.merge(df1, nb_stats, on='neighbourhood', how='left')

df_final1 = pd.merge(df_final1, boro_stats, left_on='borough', right_on='neighbourhood group', how='left')

df_final1 = pd.merge(df_final1, rt_stats, left_on='room_type', right_on='room type', how='left')

df_final1 = pd.merge(df_final1, nb_extra_stats, on='neighbourhood', how='left')
# Check if there are nulls
df_final1.isnull().sum()

#### INTEGRATION STEPS:
#### Step 1: Created neighbourhood-level Airbnb stats (mean price, density)
#### Step 2: Created borough-level Airbnb stats (price, availability, reviews)
#### Step 3: Created room type stats (avg reviews per room type)
#### Step 4: Created extra neighbourhood stats (availability, min nights)
#### Step 5: Merged all stats into Craigslist data
#### Step 6: Filled missing values using borough averages
#### Step 7: Calculated rent_vs_airbnb_ratio

#### FINAL INTEGRATED DATASET:
#### Total rows: 2065
#### Total columns: 18
#### Columns added: airbnb_price_nb, airbnb_density_nb, airbnb_price_boro, airbnb_density_boro, room type, avg_reviews, avg_reviews_per_month, avg_availability_365, avg_min_nights, avg_review_rate, rent_vs_airbnb_ratio

### Dealing with the resulted nulls

In [ ]:
df_final1['airbnb_price_nb'] = df_final1['airbnb_price_nb'].fillna(df_final1['airbnb_price_boro']) #in case there is a neighbourhood mismatch between

df_final1['airbnb_density_nb'] = df_final1['airbnb_density_nb'].fillna(df_final1['airbnb_density_boro'])

df_final1['room type'] = df_final1['room type'].fillna('Entire home/apt')

entire_home_avg = rt_stats.loc[rt_stats['room type'] == 'Entire home/apt', 'avg_reviews'].values[0]

df_final1['avg_reviews'] = df_final1['avg_reviews'].fillna(entire_home_avg)

df_final1['avg_reviews_per_month'] = df_final1['avg_reviews_per_month'].fillna(df_final1['boro_avg_rpm'])

df_final1['avg_availability_365'] = df_final1['avg_availability_365'].fillna(df_final1['boro_avg_avail'])

df_final1['avg_min_nights'] = df_final1['avg_min_nights'].fillna(df_final1['boro_avg_min_n'])

df_final1['avg_review_rate'] = df_final1['avg_review_rate'].fillna(df_final1['boro_avg_rate'])


### Drop columns and Cleanup

In [ ]:
df_final1 = df_final1.drop(columns=['boro_avg_avail', 'boro_avg_min_n', 'boro_avg_rate'])
# 1. Convert the 5-night price to a 1-night price
daily_airbnb = df_final1['airbnb_price_boro'] / 5 # to account for the fact that airbnb listings are per 5 nights so this gets the daily rate

# 2. Calculate the ratio (Monthly Rent / Daily Airbnb Price)
df_final1['rent_vs_airbnb_ratio'] = df_final1['price'] / daily_airbnb # this shows the ratio of airbnb to cover rent, meaning if this is = 10
# the host would have to rent it on airbnb for 10 days to cover 1 month of rent

df_final1 = df_final1.drop(columns=['neighbourhood group'])

print("Data Integrated")
print(f"Final Count: {len(df_final1)} rows.")
df_final1.to_csv('airbnb_with_rental_data1.csv', index=False)
df_final1.isnull().sum()

### KEY INTEGRATION DECISIONS & RATIONALE:
#### Used neighbourhood-level Airbnb stats for precise matching
#### Filled missing neighbourhoods with borough averages (pragmatic fallback)
#### Converted Unknown room types to 'Entire home/apt' (conservative assumption)
#### Calculated rent_vs_airbnb_ratio = monthly rent / daily Airbnb rate
#### Shows how many Airbnb days needed to cover 1 month of rent


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 4))
fig.suptitle("Data Integration Quality Check", fontweight='bold')

axes[0].hist(df_final1['rent_vs_airbnb_ratio'].dropna(), bins=50, color='steelblue', edgecolor='black', alpha=0.7)
axes[0].set_xlabel('Rent vs Airbnb Ratio')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Profitability Ratio Distribution')
axes[0].axvline(df_final1['rent_vs_airbnb_ratio'].median(), color='red', linestyle='--', label=f'Median: {df_final1["rent_vs_airbnb_ratio"].median():.1f}')
axes[0].legend()

completeness = df_final1.groupby('borough').size()
axes[1].bar(completeness.index, completeness.values, color='coral')
axes[1].set_xlabel('Borough')
axes[1].set_ylabel('Number of Listings')
axes[1].set_title('Listings per Borough')
axes[1].tick_params(axis='x', rotation=20)

# Questions 

## Q1. How does the average Airbnb listing price relate to the amount of rent listings accross boroughs.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

In [ ]:
# 1. Data Prep
df = pd.read_csv('airbnb_with_rental_data1.csv')
q1 = df.groupby('borough').agg(
    avg_price=('airbnb_price_boro', 'mean'),
    rent_count=('price', 'count')
).reset_index()

# 2. Setup
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Q1 · Airbnb Price vs. Rent Listing Count by Borough", fontweight='bold')
x = range(len(q1))

# Plot 1
ax1_twin = ax1.twinx()
ax1.bar([i - 0.17 for i in x], q1['avg_price'], 0.35, label='Price', color='steelblue')
ax1_twin.bar([i + 0.17 for i in x], q1['rent_count'], 0.35, label='Count', color='coral', alpha=0.8)

ax1.set_xticks(x)
ax1.set_xticklabels(q1['borough'], rotation=20, ha='right')
ax1.set_ylabel('Avg Airbnb Price ($)', color='steelblue')
ax1_twin.set_ylabel('Rent Listing Count', color='coral')
ax1.set_title('Price & Rent')

#Plot 2
ax2_twin = ax2.twinx()
ax2.scatter(q1['borough'], q1['avg_price'], color='steelblue', s=100, zorder=3)
ax2_twin.vlines(q1['borough'], 0, q1['rent_count'], color='coral', alpha=0.5, lw=4)
ax2_twin.scatter(q1['borough'], q1['rent_count'], color='coral', s=100, zorder=3)

ax2.set_xticks(x)
ax2.set_xticklabels(q1['borough'], rotation=20, ha='right')
ax2.set_ylabel('Avg Airbnb Price ($)', color='steelblue')
ax2_twin.set_ylabel('Rent Listing Count', color='coral')
ax2.set_title('Price & Rent')

plt.tight_layout()
plt.show()
boro_analysis = df_final1.groupby('borough').agg({
    'airbnb_price_boro':  'mean',
    'airbnb_density_boro': 'mean'
}).reset_index()

corr_q1 = boro_analysis['airbnb_price_boro'].corr(boro_analysis['airbnb_density_boro'])
print(f"Pearson Correlation Coefficient: {corr_q1:.4f}")


    ## Interpreting the plots: 
    #### Volume Disparity: Rent listing counts show a massive spread, from under 50 in Staten Island to over 750 in Brooklyn.
    #### The difference in average prices is very slight and pretty much inconsequential.
    
    ## Answer:
    #### There seems to be weak negative correlation between Airbnb average price and amount of rent listings.
    
    ## Insights
    #### the prices of Airbnb listings are extremely close when compared by average-per-borough and the number of rentings listed varies wildly with brookly and manhattan having significantly more listings than the other boroughs but it does not seem to effect pricing at all.

## Q2. Does the rent-vs-Airbnb ratio vary across roomtypes?

In [ ]:
q2 = df_final1.groupby('room_type')['rent_vs_airbnb_ratio'].mean().reset_index()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Q2: Rent vs Airbnb Ratio by Room Type", fontweight='bold')

sns.barplot(data=q2, x='room_type', y='rent_vs_airbnb_ratio', ax=axes[0])
axes[0].set_xlabel('Room Type')
axes[0].set_ylabel('Avg Rent vs Airbnb Ratio')
axes[0].set_title('Average Ratio per Room Type')
axes[0].tick_params(axis='x', rotation=15)

sns.boxplot(data=df_final1, x='room_type', y='rent_vs_airbnb_ratio', ax=axes[1])
axes[1].set_xlabel('Room Type')
axes[1].set_ylabel('Rent vs Airbnb Ratio')
axes[1].set_title('Distribution of Ratio per Room Type')
axes[1].tick_params(axis='x', rotation=15)

plt.tight_layout()
plt.savefig('q2.png', dpi=150)
plt.show()

df_q2 = df_final1.dropna(subset=['room type', 'rent_vs_airbnb_ratio'])
q2_stats = df_q2.groupby('room type')['rent_vs_airbnb_ratio'].agg(['mean', 'std', 'median', 'min', 'max'])
print(q2_stats)
print(f"\nOverall Mean: {df_q2['rent_vs_airbnb_ratio'].mean():.4f}")
print(f"Overall Std Dev: {df_q2['rent_vs_airbnb_ratio'].std():.4f}")

### Interpreting the plots: 
#### Entire home/apt has the highest average rent to Airbnb ratio which makes sense considering the type of customer that is looking to rent an Entire home is likely a customer that wants a long term contract, there also seems to be a much more significant amount of outliers in Entire home/apt than the other room types which indicates that some rentals are extremely pricy.

### Answer:
#### The data showcases a significant variation in the rent to Airbnb ratio across different room types with smaller units having higher margins hotel rooms have higher margin than entire homes and private rooms have higher margins than both and shared rooms have higher margins than all.

### Insights
#### the smaller the unit the more likely it is that the customer is not staying in it for long which therefore means that the customer is willing to incur higher cost.

# Q3. Is the average Airbnb minimum nights per neighbourhood related to the average rent listings count?

In [ ]:
neighbourhood_data = df_final1[~df_final1['neighbourhood'].isin(['Manhattan', 'Brooklyn', 'Queens', 'Bronx', 'Staten Island'])]

q3 = neighbourhood_data.groupby('neighbourhood').agg(
    avg_min_nights=('avg_min_nights', 'mean'),
    rent_count=('price', 'count')
).reset_index()

# Get top 10 by rent count for cleaner visualization
q3_top10 = q3.nlargest(10, 'rent_count')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Q3: Airbnb Min Nights vs Rent Count by Neighbourhood (Top 10)", fontweight='bold')

# Scatter plot
ax = axes[0]
sns.scatterplot(data=q3_top10, x='avg_min_nights', y='rent_count', s=100, color='teal', ax=ax)
for _, r in q3_top10.iterrows():
    ax.annotate(r['neighbourhood'], (r['avg_min_nights'], r['rent_count']), textcoords='offset points', xytext=(4, 4), fontsize=8)
ax.set_xlabel('Avg Min Nights')
ax.set_ylabel('Rent Listing Count')
ax.set_title('Scatter: Min Nights vs Rent Count')

# Dual axis bar chart
x = range(len(q3_top10))
w = 0.35
bars1 = axes[1].bar([i - w/2 for i in x], q3_top10['avg_min_nights'], w, label='Avg Min Nights', color='teal')
ax2 = axes[1].twinx()
bars2 = ax2.bar([i + w/2 for i in x], q3_top10['rent_count'], w, label='Rent Count', color='salmon', alpha=0.8)
axes[1].set_xticks(x)
axes[1].set_xticklabels(q3_top10['neighbourhood'], rotation=45, ha='right', fontsize=8)
axes[1].set_ylabel('Avg Min Nights', color='teal')
ax2.set_ylabel('Rent Listing Count', color='salmon')
axes[1].set_title('Dual Axis: Min Nights & Rent Count')
axes[1].legend([bars1, bars2], ['Avg Min Nights', 'Rent Count'], fontsize=8, loc='upper left')

plt.tight_layout()
plt.savefig('q3.png', dpi=150)
plt.show()

nb_data = df_final1.groupby('neighbourhood').agg({
    'avg_min_nights': 'mean',
    'airbnb_density_nb': 'mean'
}).reset_index().dropna()

corr_q3_pearson = nb_data['airbnb_density_nb'].corr(nb_data['avg_min_nights'])

print(f"Correlation: {corr_q3_pearson:.4f}")

## Interpreting the plots: 
#### The plots showcase that most neighbourhoods have similar minimum requirements between 5-7 with some outliers.

## Answer:
#### The charts make it exceedingly clear that there is no significant correlation between the amount of listings and the minimum nights requirements, some neighbourhoods have relatively average listing counts while having high minimum requirements and some show the opposite with hight listing counts and moderate minimum requirements.

## Insights
#### in the 10 largest neighbourhoods (in terms of rent listings) there seems to be some variance in minimum requirements but not due to the amount of listings in a given neighbourhood, there may be other unaccounted variables causing this variance.

# Q4. Does the distribution of room types shift across boroughs?

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Q4 · Room Type Distribution by Borough ", fontsize=13, fontweight='bold')

q4_counts = df_final1.groupby(['borough', 'room_type']).size().reset_index(name='count')
q4_pivot = q4_counts.pivot(index='borough', columns='room_type', values='count').fillna(0)
q4_pivot_pct = q4_pivot.div(q4_pivot.sum(axis=1), axis=0) * 100

ax = axes[0]
q4_pivot_pct.plot(kind='bar', stacked=True, ax=ax, colormap='Set3', edgecolor='white')
ax.set_ylabel('Percentage (%)'); ax.set_xlabel('Borough')
ax.set_title('100% Stacked Bar: Room Type Share')
ax.tick_params(axis='x', rotation=20)
ax.legend(title='Room Type', fontsize=8, bbox_to_anchor=(1, 1))


ax = axes[1]
sns.heatmap(q4_pivot, annot=True, fmt='.0f', cmap='YlOrRd',
            linewidths=0.5, ax=ax)
ax.set_title('Heatmap: Listing Count by Borough & Room Type')
ax.set_xlabel('Room Type'); ax.set_ylabel('Borough')
ax.tick_params(axis='x', rotation=15)


plt.tight_layout(); plt.savefig('q4.png', dpi=150); plt.show()


room_boro = pd.crosstab(df_final1['borough'], df_final1['room type'])
room_boro_pct = room_boro.div(room_boro.sum(axis=1), axis=0) * 100

print(room_boro_pct)


## Interpreting the plots: 
#### The overwhelming majority of listings are Entire home/apt with private rooms as a distant second and shared rooms as an even more distant third and pretty much no hotel rooms.

## Answer:
#### The distribution of room types does not seem to shift by borough with all boroughs having Entire home/apt and Private rooms as their first and second highest share of room types respectively.

## Insights
#### Non Airbnb renting is expectedly much more focused on Entire homes and private rooms than other room types because because most renters are middle income families that are looking for Entire homes.

# Q5. Is the average Airbnb rating indicative of the average price of rent accross boroughs?

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Q5 · Avg Airbnb Rating vs. Avg Rent Price by Borough", fontsize=13, fontweight='bold')

q5 = df_final1.groupby('borough').agg(
    avg_rating=('avg_review_rate', 'mean'),
    avg_rent=('price', 'mean')
).reset_index()

#plot 1
ax = axes[0]
sns.regplot(data=q5, x='avg_rating', y='avg_rent',
            scatter_kws={'s': 80, 'color': 'mediumpurple'},
            line_kws={'color': 'darkred'}, ax=ax)
for _, r in q5.iterrows():
    ax.annotate(r['borough'], (r['avg_rating'], r['avg_rent']),
                textcoords='offset points', xytext=(4, 4), fontsize=8)
ax.set_xlabel('Avg Airbnb Rating'); ax.set_ylabel('Avg Rent Price ($)')
ax.set_title('Scatter + Regression: Rating vs. Rent')

ax = axes[1]
x = range(len(q5))
w = 0.35
bars1 = ax.bar([i - w/2 for i in x], q5['avg_rating'], w,
               label='Avg Rating', color='mediumpurple')
ax2 = ax.twinx()
bars2 = ax2.bar([i + w/2 for i in x], q5['avg_rent'], w,
                label='Avg Rent ($)', color='goldenrod', alpha=0.85)
ax.set_xticks(list(x)); ax.set_xticklabels(q5['borough'], rotation=20, ha='right')
ax.set_ylabel('Avg Airbnb Rating'); ax2.set_ylabel('Avg Rent ($)')
ax.set_title('Dual-Axis: Rating & Rent per Borough')
ax.legend([bars1, bars2], ['Avg Rating', 'Avg Rent ($)'], fontsize=8, loc='upper left')

plt.tight_layout(); plt.savefig('q5.png', dpi=150); plt.show()

boro_rating = df_final1.groupby('borough').agg({
    'avg_review_rate': 'mean',
    'price': 'mean',
    'airbnb_density_boro': 'mean'
}).reset_index()

corr_q5 = boro_rating['price'].corr(boro_rating['avg_review_rate'])
print(f"Correlation: {corr_q5:.4f}")

## Interpreting the plots: 
#### The scatter plot with a regression line shows a positive slope, albeit not extremely steep. The points for the five boroughs suggests that as the average Airbnb review rating goes up, the average rent price tends to go up as well. Manhattan stands out with both the highest rating and the highest rent, while the Bronx has the lowest of both.

## Answer:
#### Yes, there is a moderate positive correlation (approx. 0.45) between a borough's average Airbnb rating and it's average rent price.

## Insights
#### It appear that neighborhoods that hosts find more desirable (higher ratings) are also more desirable for long-term renters, driving up prices in both markets. A borough's reputation, quality, and ammenities likely benifit both short-term and long-term rental markets similiarly.

# Q6. Are some boroughs more or less accepting of airbnb than rent?

In [ ]:
bnb_avail = df_final1.groupby('borough')['avg_availability_365'].sum()
rent_counts = df_final1.groupby('borough')['pid'].nunique()

bnb_avail_pct = (bnb_avail / bnb_avail.sum()) * 100
rent_pct = (rent_counts / rent_counts.sum()) * 100

q6_alternative = pd.DataFrame({
    'borough': bnb_avail.index,
    'airbnb_avail_pct': bnb_avail_pct.values,
    'rent_pct': rent_pct.values
})
q6_alternative['dominance'] = q6_alternative['airbnb_avail_pct'] - q6_alternative['rent_pct']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Q6: Airbnb Availability vs Rent Market Share", fontweight='bold')

x = np.arange(len(q6_alternative))
axes[0].bar(x - 0.2, q6_alternative['airbnb_avail_pct'], 0.4, label='Airbnb Availability %', color='steelblue')
axes[0].bar(x + 0.2, q6_alternative['rent_pct'], 0.4, label='Rent Listings %', color='coral')
axes[0].set_xticks(x)
axes[0].set_xticklabels(q6_alternative['borough'], rotation=20)
axes[0].set_ylabel('Percentage of total market (%)')
axes[0].legend()

colors = ['steelblue' if v >= 0 else 'coral' for v in q6_alternative['dominance']]
axes[1].barh(q6_alternative['borough'], q6_alternative['dominance'], color=colors)
axes[1].axvline(0, color='black', lw=1, ls='--')
axes[1].set_title('Airbnb Dominance (Availability vs Rent Listings)')

plt.tight_layout()
plt.show()

bnb_density = df_final1.groupby('borough')['airbnb_density_boro'].first()
rent_counts = df_final1.groupby('borough')['pid'].nunique()

# Calculate shares
bnb_share = (bnb_density / bnb_density.sum()) * 100
rent_share = (rent_counts / rent_counts.sum()) * 100

# Correlation
correlation, p_value = pearsonr(bnb_share, rent_share)

print(f"Correlation: {correlation:.4f}")


## Interpreting the plots: 
#### The correlation of 0.8938 indicates a very strong positive relationship between a borough's share of Airbnb listings and its share of rent listings. When one goes up, the other goes up proportionally.This means boroughs are not "choosing" Airbnb over rent or vice versa. Instead, boroughs simply have larger or smaller housing markets overall. Manhattan has the most of both, Staten Island has the least of both. The proportions stay roughly the same across all boroughs.

## Answer:
#### No, the correlation of 0.89 between Airbnb density share and rent share shows a very strong positive relationship. Boroughs with a higher share of Airbnb listings also have a similarly higher share of rent listings. There is no evidence that some boroughs "accept" Airbnb more than others relative to their rental market size.

## Insights
#### The high correlation suggests that Airbnb presence scales with the overall housing market size of each borough, not with any borough-specific preference for short-term rentals. Manhattan and Brooklyn dominate both markets proportionally, while Staten Island has the smallest share of both. This indicates Airbnb is not "replacing" rent in any borough - the two markets grow together..

# Q7. Is the amount of available airbnb listings related to the amount of available rent listings?

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Q7 · Available Airbnb Listings vs. Available Rent Listings by Borough", fontsize=13, fontweight='bold')

q7 = df_final1.groupby('borough').agg(
    avg_airbnb_avail=('avg_availability_365', 'mean'),
    rent_count=('price', 'count')
).reset_index()

ax = axes[0]
sns.scatterplot(data=q7, x='avg_airbnb_avail', y='rent_count',
                s=120, color='seagreen', ax=ax)
for _, r in q7.iterrows():
    ax.annotate(r['borough'], (r['avg_airbnb_avail'], r['rent_count']),
                textcoords='offset points', xytext=(4, 4), fontsize=8)
ax.set_xlabel('Avg Airbnb Availability (days/year)')
ax.set_ylabel('Rent Listing Count')
ax.set_title('Scatter: Airbnb Availability vs. Rent Count')

ax = axes[1]
q7_sorted = q7.sort_values('rent_count')
ax.plot(q7_sorted['borough'], q7_sorted['rent_count'],
        marker='o', label='Rent Count', color='tomato')
ax2 = ax.twinx()
ax2.plot(q7_sorted['borough'], q7_sorted['avg_airbnb_avail'],
         marker='s', linestyle='--', label='Avg Airbnb Avail', color='seagreen')
ax.set_ylabel('Rent Listing Count', color='tomato')
ax2.set_ylabel('Avg Availability (days)', color='seagreen')
ax.set_title('Trend Line: Rent Count vs. Airbnb Availability')
ax.tick_params(axis='x', rotation=20)
h1, l1 = ax.get_legend_handles_labels()
h2, l2 = ax2.get_legend_handles_labels()
ax.legend(h1 + h2, l1 + l2, fontsize=8, loc='upper left')

plt.tight_layout(); plt.savefig('q7.png', dpi=150); plt.show()

boro_inventory = df_final1.groupby('borough').agg({
    'airbnb_density_boro': 'mean',
    'pid': 'count'
}).reset_index()
boro_inventory.columns = ['borough', 'airbnb_count', 'rent_count']

corr_q7 = boro_inventory['airbnb_count'].corr(boro_inventory['rent_count'])
print(f" Correlation: {corr_q7:.4f}")

## Interpreting the plots: 
#### The scatter plot shows a very strong, clear positive linear relationship. Boroughs with more Airbnb listings (like Brooklyn and Manhattan) also have significantly more rent listings. Boroughs with fewer Airbnb listings (like Staten Island and the Bronx) have far fewer rental properties listed.

## Answer:
#### Yes, there is a very strong positive correlation (approx. 0.89). Where there are many Airbnb listings, there tends to be many rent listings as well.
## Insights
#### This suggest that the overall size and density of the housing market in a borough is the main factor. A borough with a large supply of housing overall (both for sale and for long-term rent) will also have a large supply of properties that can be used for short-term renting. It doesn't necessarily mean one market is "taking over" the other, but rather they grow together.

# Q8. Does the Airbnb listing count per borough relate to the average rent price?

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Q8 · Airbnb Listing Count vs. Avg Rent Price by Borough", fontsize=13, fontweight='bold')

q8 = df_final1.groupby('borough').agg(
    airbnb_count=('airbnb_density_boro', 'mean'),
    avg_rent=('price', 'mean')
).reset_index()


ax = axes[0]
sns.regplot(data=q8, x='airbnb_count', y='avg_rent',
            scatter_kws={'s': 80, 'color': 'darkorange'},
            line_kws={'color': 'navy'}, ax=ax)
for _, r in q8.iterrows():
    ax.annotate(r['borough'], (r['airbnb_count'], r['avg_rent']),
                textcoords='offset points', xytext=(4, 4), fontsize=8)
ax.set_xlabel('Airbnb Listing Count'); ax.set_ylabel('Avg Rent Price ($)')
ax.set_title('Scatter + Regression: Airbnb Count vs. Rent')


ax = axes[1]
scatter = ax.scatter(q8['borough'], q8['avg_rent'],
                     s=q8['airbnb_count'] / q8['airbnb_count'].max() * 1500,
                     alpha=0.6, color='darkorange', edgecolors='black')
ax.set_ylabel('Avg Rent Price ($)'); ax.set_xlabel('Borough')
ax.set_title('Bubble Chart: Rent Price (bubble ∝ Airbnb Count)')
ax.tick_params(axis='x', rotation=20)

plt.tight_layout(); plt.savefig('q8.png', dpi=150); plt.show()

q8_data = df_final1.groupby('borough').agg({
    'airbnb_density_boro': 'first',
    'price': 'mean',
    'pid': 'count'
}).reset_index()

corr_q8 = q8_data['airbnb_density_boro'].corr(q8_data['price'])
print(f"Correlation: {corr_q8:.4f}")

## Interpreting the plots: 
#### The scatter plot with a regression line shows a strong, positive correlation. As the Airbnb listing count in a borough goes up, the average rent price also goes up. Manhattan and Brooklyn have the highest numbers for both, while the Bronx and Staten Island have the lowest.

## Answer:
#### Yes, there is a very strong positive correlation (approx. 0.87) between the number of Airbnb listings and the average rent price in a borough.

## Insights
#### This high correlation is likeley driven by a common, unobserved factor: the overall 'desirability' and economic vitality of a borough. More desirable, vibrant areas attract both more short-term visitors (more Airbnbs) and more long-term residents, which drives up overall rent prices due to high demand.

# Q9. Is the average number of Airbnb reviews per borough related to the average rent price?

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Q9 · Avg Airbnb Reviews per Borough vs. Avg Rent Price", fontsize=13, fontweight='bold')

q9 = df_final1.groupby('borough').agg(
    avg_rpm=('avg_reviews_per_month', 'mean'),
    avg_rent=('price', 'mean')
).reset_index()

ax = axes[0]
sns.regplot(data=q9, x='avg_rpm', y='avg_rent',
            scatter_kws={'s': 80, 'color': 'mediumseagreen'},
            line_kws={'color': 'firebrick'}, ax=ax)
for _, r in q9.iterrows():
    ax.annotate(r['borough'], (r['avg_rpm'], r['avg_rent']),
                textcoords='offset points', xytext=(4, 4), fontsize=8)
ax.set_xlabel('Avg Reviews per Month'); ax.set_ylabel('Avg Rent Price ($)')
ax.set_title('Scatter + Regression: Reviews vs. Rent')

ax = axes[1]
q9_sorted = q9.sort_values('avg_rent', ascending=False)
color_vals = q9_sorted['avg_rpm']
norm = plt.Normalize(color_vals.min(), color_vals.max())
colors = plt.cm.RdYlGn(norm(color_vals))
bars = ax.bar(q9_sorted['borough'], q9_sorted['avg_rent'], color=colors)
sm = plt.cm.ScalarMappable(cmap='RdYlGn', norm=norm)
sm.set_array([])
plt.colorbar(sm, ax=ax, label='Avg Reviews/Month')
ax.set_ylabel('Avg Rent Price ($)'); ax.set_xlabel('Borough')
ax.set_title('Bar: Rent Price (coloured by Reviews/Month)')
ax.tick_params(axis='x', rotation=20)

plt.tight_layout(); plt.savefig('q9.png', dpi=150); plt.show()

q9_data = df_final1.groupby('borough').agg({
    'boro_avg_rpm': 'first',
    'price': 'mean'
}).reset_index()

corr_matrix_q9 = df_final1[['price', 'boro_avg_rpm', 'avg_review_rate', 'avg_availability_365']].corr()
corr_q9 = q9_data['price'].corr(q9_data['boro_avg_rpm'])

print("Correlation Matrix:")
print(corr_matrix_q9)
print(f"Price vs Reviews/Month: {corr_q9:.4f}")

## Interpreting the plots: 
#### The scatter plot shows a negative trend, but the regression line is relativly flat, and the points are clustered. The correlation is weak and negative (approx. -0.36). Manhattan has low average reviews per month but very high rent, while the Bronx has high average reviews but low rent. There is no clear pattern.

## Answer:
#### No, there does not appear to be a significant correlation between the average number of Airbnb reviews per month and the average rent price across boroughs.

## Insights
#### The average reviews per month is likely a measure of a listing's "turnover" rate, not its profitability or property value. Places with very high turnover (like the Bronx) might have lower nightly prices and lower rent stability. High-rent areas like Manhattan might have listing that cater to guests staying for longer periods, leading to fewer reviews per month desptie higher value.

# Q10. do availability trends influence rent prices accross neighbourhoods?

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Q10 · Airbnb Availability vs. Rent Prices across Neighbourhoods", fontsize=13, fontweight='bold')

q10 = df_final1.groupby('neighbourhood').agg(
    avg_avail=('avg_availability_365', 'mean'),
    avg_rent=('price', 'mean')
).reset_index().dropna()

ax = axes[0]
ax.scatter(q10['avg_avail'], q10['avg_rent'],
           alpha=0.5, s=30, color='slateblue')
m, b = pd.np.polyfit(q10['avg_avail'], q10['avg_rent'], 1) if hasattr(pd, 'np') \
    else (__import__('numpy').polyfit(q10['avg_avail'], q10['avg_rent'], 1))
import numpy as np
m, b = np.polyfit(q10['avg_avail'], q10['avg_rent'], 1)
ax.plot(sorted(q10['avg_avail']), [m*v+b for v in sorted(q10['avg_avail'])],
        color='red', linewidth=1.5)
ax.set_xlabel('Avg Availability (days/year)'); ax.set_ylabel('Avg Rent Price ($)')
ax.set_title('Scatter: Availability vs. Rent (All Neighbourhoods)')

ax = axes[1]
hb = ax.hexbin(q10['avg_avail'], q10['avg_rent'],
               gridsize=20, cmap='YlOrRd', mincnt=1)
plt.colorbar(hb, ax=ax, label='Neighbourhood Count')
ax.set_xlabel('Avg Availability (days/year)'); ax.set_ylabel('Avg Rent Price ($)')
ax.set_title('Hexbin Density: Availability vs. Rent')

plt.tight_layout(); plt.savefig('q10.png', dpi=150); plt.show()

q10_data = df_final1.groupby('neighbourhood').agg({
    'avg_availability_365': 'first',
    'price': 'mean'
}).reset_index().dropna()

corr_q10_pearson = q10_data['avg_availability_365'].corr(q10_data['price'])

print(f"Correlation: {corr_q10_pearson:.4f}")

## Interpreting the plots: 
#### The hexbin plot and scatter plot show a cluster of neighborhoods with an average availability between 100 and 220 days per year. For these, rent prices vary wildly, from around $2000 - $8000. The overall regression line is almost flat, and the correlation is very weak (approx. -0.36).

## Answer:
#### No, a neighborhood's average Airbnb availability (how many days a year a listing is bookable) does not meaningfully influence its average rent price.

## Insights
#### Availability might be more of a host-driven decision, based on thier personal use of the property or strategies to maximize occupancy, rather than a signal of the underlying housing market's strength. It does not appear to be a factor that renters care about or that correlates with property value trends.

# Q11. Does the price variance of Airbnb listing prices within a borough relate to that borough's average rent.

In [ ]:
BOROS_TO_EXCLUDE = ['Manhattan', 'Brooklyn', 'Bronx', 'Queens', 'Staten Island']

q11_clean = df_final1[~df_final1['neighbourhood'].isin(BOROS_TO_EXCLUDE)]

q11 = q11_clean.groupby('borough').agg(
    price_std=('airbnb_price_nb', 'std'),
    avg_rent=('price', 'mean')
).reset_index().fillna(0)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Q11: Airbnb Price Variance vs Avg Rent by Borough", fontweight='bold')

# Plot 1: Scatter with labels
ax = axes[0]
sns.regplot(data=q11, x='price_std', y='avg_rent', 
            scatter_kws={'s': 100, 'color': 'indianred'}, 
            line_kws={'color': 'darkblue'}, ax=ax)

for _, r in q11.iterrows():
    ax.annotate(r['borough'], (r['price_std'], r['avg_rent']), 
                textcoords='offset points', xytext=(5, 5), fontsize=9)

ax.set_xlabel('Airbnb Price Std Dev ($)')
ax.set_ylabel('Avg Rent Price ($)')

# Plot 2: Bar + Error Bars
ax = axes[1]
q11_sorted = q11.sort_values('avg_rent', ascending=False)
ax.bar(q11_sorted['borough'], q11_sorted['avg_rent'], 
       yerr=q11_sorted['price_std'], capsize=8, 
       color='indianred', alpha=0.8, ecolor='black')

ax.set_ylabel('Avg Rent Price ($)')
ax.tick_params(axis='x', rotation=20)

plt.tight_layout()
plt.show()

# Correlation check
corr_q11 = q11['price_std'].corr(q11['avg_rent'])
print(f"Correlation: {corr_q11:.4f}")

## Interpreting the plots: 
#### The scatter plot show a very strong negative correlation (approx. -0.60). As the standard deviation of Airbnb prices within a borough increases, the average rent price for that borough decreases. Manhattan has low price variance but very high rent, while the Bronx and Staten Island have high price variance but low rent.

## Answer:
#### Yes, there is a moderately strong negative correlation. Boroughs with a wider spread of Airbnb prices tend to have lower average rent prices.

## Insights
#### Areas with consistently high-value properties (low variance, all expensive) like Manhattan command high rents across the board. Areas with a mix of very cheap and very expensive listings (high variance) may have some very affordable rent options, pulling the borough's average rent down. Variance might be a proxy for economic inequality and housing stock diveristy.

# Q12. Does the combination of room type and average minimum nights together relate to the rent-vs-Airbnb ratio across boroughs?

In [ ]:
q12 = df_final1.dropna(subset=['room type', 'avg_min_nights', 'rent_vs_airbnb_ratio', 'borough'])
q12['min_nights_cat'] = pd.cut(q12['avg_min_nights'], bins=[0, 7, 30, 365], labels=['Short (0-7)', 'Medium (8-30)', 'Long (31-365)'])

fig, axes = plt.subplots(1, 2, figsize=(15, 6))
fig.suptitle("Q12: Room Type + Min Nights vs Profitability Ratio", fontweight='bold', fontsize=13)

# Plot 1
pivot = q12.groupby(['room type', 'min_nights_cat'])['rent_vs_airbnb_ratio'].mean().unstack()
sns.heatmap(pivot, annot=True, fmt='.2f', cmap='RdYlGn_r', ax=axes[0],
            linewidths=1, cbar_kws={'label': 'Mean Ratio'})
axes[0].set_title('Mean Profitability: Room Type × Min Nights')
axes[0].set_xlabel('Min Nights Category', fontweight='bold')
axes[0].set_ylabel('Room Type', fontweight='bold')

# Plot 2
sns.stripplot(data=q12, x='room type', y='rent_vs_airbnb_ratio', hue='min_nights_cat',
              ax=axes[1], palette='Set2', alpha=0.6, dodge=True, size=5)
axes[1].set_title('Ratio Distribution by Room Type & Stay Length')
axes[1].set_xlabel('Room Type', fontweight='bold')
axes[1].set_ylabel('Rent-vs-Airbnb Ratio', fontweight='bold')
axes[1].legend(title='Min Nights', fontsize=9)
axes[1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('q12.png', dpi=150, bbox_inches='tight')
plt.show()


groups = [g['rent_vs_airbnb_ratio'].values for _, g in q12.groupby(['room type', 'min_nights_cat'])]
corr = q12['avg_min_nights'].corr(q12['rent_vs_airbnb_ratio'])

print(f"Min Nights vs Ratio Correlation: {corr:.4f}")
print(q12.groupby(['room type', 'min_nights_cat'])['rent_vs_airbnb_ratio'].agg(['mean','std','count']))

## Interpreting the plots: 
#### The heatmap clearly shows that for every room type, the rent_vs_airbnb_ratio is significantly higher for listings with "Medium (8-30)" minimum nights compared to "Short (0-7)" minimum nights. The stripplot shows that "Shared rooms" with higher min nights have ratios clustered far above others.

## Answer:
#### Yes, the combination of room type and minimum nights has a strong effect. Listings that require a longer stay (8+ nights) generate a much higher ratio. Shared rooms further amplifies this effect, having the highest ratios of all.

## Insights
#### Longer minimum nights directly shifts the Airbnb booking from a short-term vacation rental toward a medium-term "corporate" or "temporary housing" rental. This market segment is much more profitable when compared to long-term rent. It also suggests that a more "professional," less "on-demand" hosting style leads to a different financial dynamic completly.

# Hypotheses

## Hyothesis 1: Room Type Distribution vs Borough Rent Level

In [ ]:
# Create price categories based on quantiles
q1 = df_final1['price'].quantile(0.33)
q2 = df_final1['price'].quantile(0.66)

def price_category(x):
    if x <= q1:
        return 'Budget'
    elif x <= q2:
        return 'Mid-Range'
    else:
        return 'Premium'

# Create borough rent level based on average borough price
boro_avg = df_final1.groupby('borough')['price'].mean()
boro_rent_level = boro_avg.apply(price_category)
df_final1['boro_rent_level'] = df_final1['borough'].map(boro_rent_level)

# Contingency table
contingency1 = pd.crosstab(df_final1['room_type'], df_final1['boro_rent_level'])

# Chi-square test
chi2, p, dof, expected = stats.chi2_contingency(contingency1)

# Output
print(contingency1)
print("Chi-square:", chi2)
print("P-value:", p)

# Decision
alpha = 0.05
if p < alpha:
    print("Decision: Reject H0")
    print("Conclusion: Room type distribution depends on borough rent level")
else:
    print("Decision: Fail to reject H0")
    print("Conclusion: Room type distribution is independent of borough rent level")

## Hypothesis 2: Price Category vs Borough

In [ ]:
# Create price categories
q1 = df_final1['price'].quantile(0.33)
q2 = df_final1['price'].quantile(0.66)

df_final1['price_category'] = df_final1['price'].apply(price_category)

# Contingency table
contingency2 = pd.crosstab(df_final1['borough'], df_final1['price_category'])

# Chi-square test
chi2, p, dof, expected = stats.chi2_contingency(contingency2)

# Output
print(contingency2)
print("Chi-square:", chi2)
print("P-value:", p)

# Decision
alpha = 0.05
if p < alpha:
    print("Decision: Reject H0")
    print("Conclusion: Price category depends on borough")
else:
    print("Decision: Fail to reject H0")
    print("Conclusion: Price category is independent of borough")

## Hypothesis 3: Ratio Category vs Room Type

In [ ]:
# Create ratio categories
q1 = df_final1['rent_vs_airbnb_ratio'].quantile(0.33)
q2 = df_final1['rent_vs_airbnb_ratio'].quantile(0.66)

def ratio_category(x):
    if x <= q1:
        return 'Low'
    elif x <= q2:
        return 'Medium'
    else:
        return 'High'

df_final1['ratio_category'] = df_final1['rent_vs_airbnb_ratio'].apply(ratio_category)

# Contingency table
contingency3 = pd.crosstab(df_final1['room_type'], df_final1['ratio_category'])

# Chi-square test
chi2, p, dof, expected = stats.chi2_contingency(contingency3)

# Output
print(contingency3)
print("Chi-square:", chi2)
print("P-value:", p)

# Decision
alpha = 0.05
if p < alpha:
    print("Decision: Reject H0")
    print("Conclusion: Ratio category depends on room type")
else:
    print("Decision: Fail to reject H0")
    print("Conclusion: Ratio category is independent of room type")

## Hypothesis 4: Availability Category vs Borough Rent Level

In [ ]:
# Create availability category
q1_avail = df_final1['avg_availability_365'].quantile(0.33)
q2_avail = df_final1['avg_availability_365'].quantile(0.66)
print(q1_avail)
print(q2_avail)

def availability_category(x):
    if x <= q1_avail:
        return 'Low'
    elif x <= q2_avail:
        return 'Medium'
    else:
        return 'High'

df_final1['availability_category'] = df_final1['avg_availability_365'].apply(availability_category)

# Create rent level using price quantiles
q1 = df_final1['price'].quantile(0.33)
q2 = df_final1['price'].quantile(0.66)

df_final1['rent_level'] = df_final1['price'].apply(price_category)
df_final1['price_category'] = df_final1['price'].apply(price_category)

# Contingency table
contingency4 = pd.crosstab(df_final1['availability_category'], df_final1['price_category'])

# Chi-square test
chi2, p, dof, expected = stats.chi2_contingency(contingency4)
print(contingency4)
print("Chi-square:", chi2)
print("P-value:", p)

# Decision
alpha = 0.05
if p < alpha:
    print("Decision: Reject H0")
    print("Conclusion: Availability is dependent on rent level")
else:
    print("Decision: Fail to reject H0")
    print("Conclusion: Availability is independent of rent level")